# Fine-tuning com Correções Reais — Classificador de Soja

Este notebook pega o modelo já treinado (`soja_model_final.keras`) e o **ajusta** com as fotos
que você corrigiu no app (salvas no dataset `Guguinhaxd/soja-correction`).

## Por que as correções têm peso máximo aqui

Uma correção manual não é dado qualquer — é sinal de qualidade máxima:
1. É uma foto do **seu** celular, no **seu** setup (resolve domain shift diretamente)
2. O modelo **já errou** nela → é exatamente onde ele precisa aprender
3. Você **viu o grão** e disse a classe certa → é ground truth confiável

Por isso este notebook dá **peso 2× a cada correção** na loss (o gradiente dela conta o dobro),
usa **70% correções / 30% original** no mix, e aplica **augmentation forte** nas correções
para tirar máximo proveito de poucas fotos.

O dataset original (30%) existe só pra evitar *catastrophic forgetting* — sem ele, o modelo
esquece tudo que sabia e vira uma bagunça.

### Pré-requisitos
1. `soja_model_final.keras` no Drive (gerado pelo `train.ipynb`)
2. Pelo menos ~8–10 correções por classe no dataset (quanto mais, melhor)
3. GPU ligada: `Ambiente de execução → Alterar tipo → T4 GPU`

In [ ]:
# Célula 1 — GPU, imports e login no Hugging Face
!pip install -q huggingface_hub

import os, json, glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from huggingface_hub import snapshot_download, HfApi

print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {gpus}')
assert len(gpus) > 0, 'Sem GPU! Ambiente de execução → Alterar tipo → T4'

# Pegue seu token em: https://huggingface.co/settings/tokens
# Precisa de permissão de leitura no dataset e escrita no Space.
from getpass import getpass
HF_TOKEN = getpass('Cole seu token do Hugging Face: ')

In [ ]:
# Célula 2 — Montar Drive, caminhos e classes
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT    = '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)'
TRAIN_DIR       = f'{DATASET_ROOT}/train'
TEST_DIR        = f'{DATASET_ROOT}/test'
SAVE_DIR        = '/content/drive/MyDrive'
BASE_MODEL_PATH = f'{SAVE_DIR}/soja_model_final.keras'
CORRECTIONS_REPO = 'Guguinhaxd/soja-correction'

# Mesma ordem do train.ipynb — NÃO mudar (índice = saída do modelo)
CLASS_NAMES = [
    'Broken soybeans',
    'Immature soybeans',
    'Intact soybeans',
    'Skin-damaged soybeans',
    'Spotted soybeans',
]
SHORT = {
    'Broken soybeans':       'broken',
    'Immature soybeans':     'immature',
    'Intact soybeans':       'intact',
    'Skin-damaged soybeans': 'skin-damaged',
    'Spotted soybeans':      'spotted',
}
NUM_CLASSES = len(CLASS_NAMES)

assert os.path.isfile(BASE_MODEL_PATH), f'Modelo base não encontrado: {BASE_MODEL_PATH}'
assert os.path.isdir(TRAIN_DIR), f'Dataset original não encontrado: {TRAIN_DIR}'
print('OK — modelo base e dataset original encontrados.')

In [ ]:
# Célula 3 — Baixar correções do Hugging Face e contar por classe
CORR_DIR = snapshot_download(
    repo_id=CORRECTIONS_REPO,
    repo_type='dataset',
    token=HF_TOKEN,
    local_dir='/content/correcoes',
)
print(f'Correções baixadas em: {CORR_DIR}\n')

corr_counts = {}
for cls in CLASS_NAMES:
    files = (glob.glob(os.path.join(CORR_DIR, cls, '*.jpg'))  +
             glob.glob(os.path.join(CORR_DIR, cls, '*.jpeg')) +
             glob.glob(os.path.join(CORR_DIR, cls, '*.png')))
    corr_counts[cls] = files
    print(f'{SHORT[cls]:>14}: {len(files):>3} correções')

total_corr = sum(len(v) for v in corr_counts.values())
print(f'\nTotal de correções: {total_corr}')
assert total_corr >= 5, 'Poucas correções! Colete mais fotos pelo app antes de re-treinar.'
if total_corr < NUM_CLASSES * 8:
    print('⚠️  Recomendado ~8+ por classe. Dá pra rodar mesmo assim, mas quanto mais, melhor.')

In [ ]:
# Célula 3.5 — Recortar cada grão (MESMA lógica do app.py)
# Por que: o modelo classifica o grão RECORTADO (preenchendo o quadro), igual ao dataset
# original. As correções precisam ter o mesmo enquadramento, senão treino != inferência.
# As 57 primeiras fotos foram salvas inteiras; uploads novos também podem vir inteiros.
# Esta célula normaliza TODAS recortando do mesmo jeito. É idempotente: se a imagem já
# está recortada (grão preenche >95% do quadro), ela é mantida como está.
import cv2

def crop_single_grain(arr):
    h, w    = arr.shape[:2]
    gray    = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    thresh  = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    thresh  = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=1)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return arr
    largest = max(contours, key=cv2.contourArea)
    if not (0.02 < cv2.contourArea(largest) / (h * w) < 0.95):
        return arr
    x, y, bw, bh = cv2.boundingRect(largest)
    pad = max(15, int(min(bw, bh) * 0.12))
    return arr[max(0, y-pad):min(h, y+bh+pad), max(0, x-pad):min(w, x+bw+pad)]

from PIL import Image as PILImage
CROP_DIR = '/content/correcoes_cropped'
cropped_counts = {}
for cls in CLASS_NAMES:
    out_dir = os.path.join(CROP_DIR, cls)
    os.makedirs(out_dir, exist_ok=True)
    saved = []
    for fp in corr_counts[cls]:
        try:
            arr  = np.array(PILImage.open(fp).convert('RGB'))
            crop = crop_single_grain(arr)
            dst  = os.path.join(out_dir, os.path.basename(fp))
            PILImage.fromarray(crop).save(dst, format='JPEG', quality=95)
            saved.append(dst)
        except Exception as e:
            print(f'  pulei {os.path.basename(fp)}: {e}')
    cropped_counts[cls] = saved
    print(f'{SHORT[cls]:>14}: {len(saved):>3} recortadas')

# A partir daqui o pipeline usa as versões recortadas
corr_counts = cropped_counts
print('\nRecorte concluído — todas as correções no mesmo enquadramento do modelo.')

In [ ]:
# Célula 5 — Pipelines tf.data com peso nas correções
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
AUTOTUNE    = tf.data.AUTOTUNE

# Com mais dados (>200 correções), o peso pode ser menor — cada exemplo já aparece
# mais vezes naturalmente. Com <100 fotos use 2.0; com >200 use 1.5.
n_total_corr = len(ct_paths)
CORR_WEIGHT  = 2.0 if n_total_corr < 150 else 1.5
ORIG_WEIGHT  = 1.0
print(f'Correções de treino: {n_total_corr}  →  CORR_WEIGHT={CORR_WEIGHT}')

# Augmentation FORTE para as correções.
# São as fotos reais do setup do usuário — variamos bastante para cobrir ângulos/luz.
aug_strong = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(height_factor=0.12, width_factor=0.12),
    layers.RandomBrightness(0.25),
    layers.RandomContrast(0.25),
], name='aug_strong')

# Augmentation LEVE para o dataset original (só pra evitar catastrophic forgetting).
aug_light = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomBrightness(0.05),
], name='aug_light')

def load_corr_raw(path, label):
    img   = tf.io.read_file(path)
    img   = tf.io.decode_image(img, channels=3, expand_animations=False)
    img   = tf.image.resize(img, IMG_SIZE)
    img   = tf.cast(img, tf.float32)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

# Dataset original
orig_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, class_names=CLASS_NAMES, image_size=IMG_SIZE,
    batch_size=None, label_mode='categorical', shuffle=True, seed=42,
).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)

# Dataset de correções
corr_raw = (
    tf.data.Dataset.from_tensor_slices((ct_paths, ct_labels))
    .shuffle(len(ct_paths), seed=42, reshuffle_each_iteration=True)
    .map(load_corr_raw, num_parallel_calls=AUTOTUNE)
)

# Padrão: .batch() → .map(aug) → .unbatch() para augmentation Keras em elementos individuais.
corr_stream = (
    corr_raw.repeat()
    .batch(BATCH_SIZE)
    .map(lambda x, y: (aug_strong(x, training=True), y,
                       tf.ones([tf.shape(x)[0]]) * CORR_WEIGHT), num_parallel_calls=AUTOTUNE)
    .unbatch()
)

orig_stream = (
    orig_raw.repeat()
    .batch(BATCH_SIZE)
    .map(lambda x, y: (aug_light(x, training=True), y,
                       tf.ones([tf.shape(x)[0]]) * ORIG_WEIGHT), num_parallel_calls=AUTOTUNE)
    .unbatch()
)

# 70% correções / 30% original.
mixed = (
    tf.data.Dataset.sample_from_datasets(
        [corr_stream, orig_stream], weights=[0.7, 0.3], seed=42
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Validação: suas fotos sem augmentation
def load_val(path, label):
    img   = tf.io.read_file(path)
    img   = tf.io.decode_image(img, channels=3, expand_animations=False)
    img   = tf.image.resize(img, IMG_SIZE)
    img   = tf.cast(img, tf.float32)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

corr_val = (
    tf.data.Dataset.from_tensor_slices((cv_paths, cv_labels))
    .map(load_val, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

STEPS_PER_EPOCH = max(60, (len(ct_paths) * 8) // int(BATCH_SIZE * 0.7))
print(f'Steps por época: {STEPS_PER_EPOCH}  (~{STEPS_PER_EPOCH * int(BATCH_SIZE * 0.7) / max(1,len(ct_paths)):.0f}× por correção)')

In [ ]:
# Célula 5 — Pipelines tf.data com peso nas correções
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
AUTOTUNE    = tf.data.AUTOTUNE
CORR_WEIGHT = 2.0   # cada correção vale 2 exemplos na loss
ORIG_WEIGHT = 1.0

# Augmentation FORTE para as correções.
# São poucas fotos → variamos bastante para extrair o máximo de cada uma.
aug_strong = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(height_factor=0.12, width_factor=0.12),
    layers.RandomBrightness(0.25),
    layers.RandomContrast(0.25),
], name='aug_strong')

# Augmentation LEVE para o dataset original.
# Está aqui só pra evitar catastrophic forgetting — não precisa ser forte.
aug_light = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomBrightness(0.05),
], name='aug_light')

def load_corr_raw(path, label):
    img   = tf.io.read_file(path)
    img   = tf.io.decode_image(img, channels=3, expand_animations=False)
    img   = tf.image.resize(img, IMG_SIZE)
    img   = tf.cast(img, tf.float32)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

# Dataset original (label já é one-hot via image_dataset_from_directory)
orig_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, class_names=CLASS_NAMES, image_size=IMG_SIZE,
    batch_size=None, label_mode='categorical', shuffle=True, seed=42,
).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)

# Dataset de correções
corr_raw = (
    tf.data.Dataset.from_tensor_slices((ct_paths, ct_labels))
    .shuffle(len(ct_paths), seed=42, reshuffle_each_iteration=True)
    .map(load_corr_raw, num_parallel_calls=AUTOTUNE)
)

# Aplica augmentation em lotes e adiciona o sample_weight.
# Padrão: .batch() → .map(aug) → .unbatch() — necessário porque as camadas Keras
# de augmentation esperam tensores 4D (batch, H, W, C).
corr_stream = (
    corr_raw.repeat()
    .batch(BATCH_SIZE)
    .map(lambda x, y: (aug_strong(x, training=True), y,
                       tf.ones([tf.shape(x)[0]]) * CORR_WEIGHT), num_parallel_calls=AUTOTUNE)
    .unbatch()
)

orig_stream = (
    orig_raw.repeat()
    .batch(BATCH_SIZE)
    .map(lambda x, y: (aug_light(x, training=True), y,
                       tf.ones([tf.shape(x)[0]]) * ORIG_WEIGHT), num_parallel_calls=AUTOTUNE)
    .unbatch()
)

# 70% correções / 30% original.
# Correções: 70% presença × 2.0 de peso na loss = ~4.7× mais influência que o original.
mixed = (
    tf.data.Dataset.sample_from_datasets(
        [corr_stream, orig_stream], weights=[0.7, 0.3], seed=42
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Validação: suas fotos sem augmentation, sem peso (acurácia real no seu domínio)
def load_val(path, label):
    img   = tf.io.read_file(path)
    img   = tf.io.decode_image(img, channels=3, expand_animations=False)
    img   = tf.image.resize(img, IMG_SIZE)
    img   = tf.cast(img, tf.float32)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

corr_val = (
    tf.data.Dataset.from_tensor_slices((cv_paths, cv_labels))
    .map(load_val, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ~8 visualizações por correção por época (via augmentation + 70% presença)
STEPS_PER_EPOCH = max(60, (len(ct_paths) * 8) // int(BATCH_SIZE * 0.7))
print(f'Steps por época: {STEPS_PER_EPOCH}')
print(f'  → cada correção vista ~{STEPS_PER_EPOCH * int(BATCH_SIZE * 0.7) / len(ct_paths):.0f}× por época (com augmentation diferente a cada vez)')

In [ ]:
# Célula 7 — Fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

# patience aumentado de 4 → 6 para dar mais chance com mais dados.
# epochs aumentado de 15 → 20 pelo mesmo motivo.
callbacks = [
    keras.callbacks.EarlyStopping(
        patience=6, restore_best_weights=True,
        monitor='val_accuracy', verbose=1),
    keras.callbacks.ModelCheckpoint(
        f'{SAVE_DIR}/soja_finetuned_best.keras',
        save_best_only=True, monitor='val_accuracy', verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        patience=3, factor=0.5, monitor='val_loss',
        min_lr=1e-7, verbose=1),
]

print('=== Fine-tuning — 70% correções / 30% original ===')
print(f'CORR_WEIGHT={CORR_WEIGHT}  |  epochs=20  |  steps/epoch={STEPS_PER_EPOCH}')
history = model.fit(
    mixed,
    validation_data=corr_val,
    epochs=20,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks,
)

In [ ]:
# Célula 7 — Fine-tuning
# LR bem baixo: ajusta sem destruir os pesos do ImageNet
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        patience=4, restore_best_weights=True,
        monitor='val_accuracy', verbose=1),
    keras.callbacks.ModelCheckpoint(
        f'{SAVE_DIR}/soja_finetuned_best.keras',
        save_best_only=True, monitor='val_accuracy', verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        patience=2, factor=0.5, monitor='val_loss',
        min_lr=1e-7, verbose=1),
]

print('=== Fine-tuning — 70% correções (peso 2×) / 30% original ===')
history = model.fit(
    mixed,
    validation_data=corr_val,
    epochs=15,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks,
)

In [ ]:
# Célula 8 — Comparar ANTES vs DEPOIS + verificar que não esqueceu o dataset original
new_loss, new_acc = model.evaluate(corr_val, verbose=0)
print(f'ANTES  — acurácia nas suas fotos: {base_acc:.1%}')
print(f'DEPOIS — acurácia nas suas fotos: {new_acc:.1%}')
delta = (new_acc - base_acc) * 100
print(f'Ganho: {delta:+.1f} pontos percentuais')

# Verificação de sanidade: o modelo NÃO pode ter esquecido o dataset original
test_ds = (
    tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, class_names=CLASS_NAMES, image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False,
    )
    .map(lambda x, y: (tf.cast(x, tf.float32), y))
    .prefetch(AUTOTUNE)
)
_, test_acc = model.evaluate(test_ds, verbose=0)
print(f'\nAcurácia no TESTE original: {test_acc:.1%}  (não pode despencar abaixo de ~70%)')

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Treino (misto)')
plt.plot(history.history['val_accuracy'], label='Validação (suas fotos)')
plt.title('Acurácia'); plt.xlabel('Época'); plt.legend(); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Loss'); plt.xlabel('Época'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Célula 9 — Salvar
# Só sobrescreve o modelo de produção se realmente melhorou nas suas fotos
# e não destruiu o dataset original.
FINETUNED_PATH = f'{SAVE_DIR}/soja_finetuned_final.keras'
model.save(FINETUNED_PATH)
print(f'Modelo ajustado salvo em: {FINETUNED_PATH}')

if new_acc >= base_acc and test_acc >= 0.70:
    model.save(f'{SAVE_DIR}/soja_model_final.keras')
    print('\n✅ Melhorou nas suas fotos e não destruiu o teste original.')
    print('   Sobrescrevi soja_model_final.keras — pronto para o Space.')
else:
    print('\n⚠️  Resultado abaixo do esperado. Mantive o soja_model_final.keras antigo.')
    print('   Colete mais correções (mais por classe) e rode de novo.')
    if test_acc < 0.70:
        print('   O modelo esqueceu demais o dataset original — pode ser que o BATCH_SIZE')
        print('   ou CORR_WEIGHT estejam muito agressivos. Tente reduzir CORR_WEIGHT para 1.5.')

In [ ]:
# Célula 10 (OPCIONAL) — Subir o modelo direto pro HF Space via API
# Evita o download/git manual do .keras (29 MB com LFS é chato).
# O mesmo token do início serve, desde que tenha permissão de escrita no Space.
SPACE_REPO = 'Guguinhaxd/soja-inspection'

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=f'{SAVE_DIR}/soja_model_final.keras',
    path_in_repo='soja_model_final.keras',
    repo_id=SPACE_REPO,
    repo_type='space',
    commit_message='fine-tune: modelo ajustado com correções reais',
)
print('✅ Modelo enviado pro Space. Ele vai rebuildar sozinho em ~2 min.')